In [1]:
from qmirt.utils.filesystem import find_project_root
import pandas as pd
import numpy as np
import opengate as gate
from pathlib import Path

from opengate.geometry.volumes import BoxVolume, RepeatParametrisedVolume, TrdVolume

In [2]:
project_root = find_project_root()
print(f"Project root: {project_root}")

Project root: /home/fanghan/Work/MGB-HMS/RPIL/QMIRT/qmirt-gate-10-sim


In [3]:
geometry_config_csv_path = (
    project_root
    / "persistent_data/cardiac_spect/spreadsheet/MDSL.excel80M10RFR.cut-plate.010.150roi.2.30pin.105ellipse_geometry_config.csv"
)
geometry_config_df = pd.read_csv(geometry_config_csv_path)

In [4]:
print(geometry_config_df.columns)

Index(['hole_center_x (mm)', 'hole_center_y (mm)', 'hole_center_z (mm)',
       'wall_thickness (mm)', 'hole_r (mm)', 'body_l (mm)',
       'body_l_corrected (mm)', 'body_inner_top (mm)',
       'body_inner_top_corrected (mm)', 'body_outer_top (mm)',
       'body_inner_bottom (mm)', 'body_outer_bottom (mm)', 'guide_l (mm)',
       'guide_inner_top (mm)', 'guide_outer_top (mm)',
       'guide_inner_bottom (mm)', 'guide_outer_bottom (mm)',
       'box_inner_size_x (mm)', 'box_inner_size_y (mm)',
       'box_inner_size_z (mm)', 'box_outer_size_x (mm)',
       'box_outer_size_y (mm)', 'box_outer_size_z (mm)', 'z_axis_angle (deg)',
       'x_axis_angle (deg)'],
      dtype='str')


In [5]:
print(
    geometry_config_df[
        [
            "body_l (mm)",
            "body_l_corrected (mm)",
            "wall_thickness (mm)",
            "body_inner_top (mm)",
            "body_inner_bottom (mm)",
            "body_inner_top_corrected (mm)",
            "body_outer_top (mm)",
            "body_outer_bottom (mm)",
        ]
    ].iloc[0]
)


body_l (mm)                      89.070206
body_l_corrected (mm)            87.070206
wall_thickness (mm)               2.000000
body_inner_top (mm)              50.000000
body_inner_bottom (mm)            2.300000
body_inner_top_corrected (mm)    48.928935
body_outer_top (mm)              52.928935
body_outer_bottom (mm)            6.300000
Name: 0, dtype: float64


In [6]:
print(geometry_config_df[["box_outer_size_x (mm)", "box_outer_size_z (mm)"]].iloc[0])
print(geometry_config_df[["box_inner_size_x (mm)", "box_inner_size_z (mm)"]].iloc[0])

box_outer_size_x (mm)    55.2
box_outer_size_z (mm)    16.0
Name: 0, dtype: float64
box_inner_size_x (mm)    51.2
box_inner_size_z (mm)    10.0
Name: 0, dtype: float64


In [113]:
hole_centers = geometry_config_df[
    ["hole_center_x (mm)", "hole_center_y (mm)", "hole_center_z (mm)"]
].to_numpy()
collimator_body_l = geometry_config_df["body_l_corrected (mm)"].to_numpy()
collimator_guide_l = geometry_config_df["guide_l (mm)"].to_numpy()
hole_r_np = geometry_config_df["hole_r (mm)"].to_numpy()
collimator_body_centers = (
    hole_centers * ((hole_r_np + collimator_body_l * 0.5) / hole_r_np)[..., np.newaxis]
)
collimator_guide_center = (
    hole_centers * ((hole_r_np - collimator_guide_l * 0.5) / hole_r_np)[..., np.newaxis]
)

collimator_container_l = collimator_body_l + collimator_guide_l + geometry_config_df["box_outer_size_z (mm)"].to_numpy()
collimator_container_r = hole_r_np + collimator_container_l * 0.5 - collimator_guide_l
collimator_container_centers = (
    hole_centers * (collimator_container_r / hole_r_np)[..., np.newaxis]
)

crystal_r_np = geometry_config_df["body_l (mm)"].to_numpy() + hole_r_np + 5.0
box_outer_r_np = geometry_config_df["box_outer_size_z (mm)"].to_numpy() * 0.5 + hole_r_np + collimator_body_l
box_outer_centers = hole_centers * ((box_outer_r_np) / hole_r_np)[..., np.newaxis]
crystal_centers = hole_centers * ((crystal_r_np) / hole_r_np)[..., np.newaxis]
# Print the hole centers shape
print(f"Hole centers shape: {hole_centers.shape}")
print(f"Collimator body centers shape: {collimator_body_centers.shape}")
print(f"Collimator guide centers shape: {collimator_guide_center.shape}")
print(f"Collimator container centers shape: {collimator_container_centers.shape}")
print(f"Crystal centers shape: {crystal_centers.shape}")
print(f"Box outer centers shape: {box_outer_centers.shape}")


Hole centers shape: (80, 3)
Collimator body centers shape: (80, 3)
Collimator guide centers shape: (80, 3)
Collimator container centers shape: (80, 3)
Crystal centers shape: (80, 3)
Box outer centers shape: (80, 3)


In [ ]:
# Plot the hole centers in 3D using plotly
import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=hole_centers[:, 0],
            y=hole_centers[:, 1],
            z=hole_centers[:, 2],
            mode="markers",
            marker=dict(size=5, color="blue", opacity=0.8),
        ),
        go.Scatter3d(
            x=collimator_body_centers[:, 0],
            y=collimator_body_centers[:, 1],
            z=collimator_body_centers[:, 2],
            mode="markers",
            marker=dict(size=5, color="red", opacity=0.8),
        ),
    ]
)
fig.add_trace(
    go.Scatter3d(
        x=crystal_centers[:, 0],
        y=crystal_centers[:, 1],
        z=crystal_centers[:, 2],
        mode="markers",
        marker=dict(size=5, color="green", opacity=0.8),
    )
)
fig.show()


In [108]:
from scipy.spatial.transform import Rotation


def make_gate_shell_box(
    sim: gate.Simulation,
    *,
    name: str,
    mother: str,
    outer_size_mm: np.ndarray,
    inner_size_mm: np.ndarray,
    spacing_size_mm: np.ndarray,
    translation_mm: list[float] | np.ndarray,
    inner_shift_mm: list[float] | np.ndarray = [0, 0, 0],
    material: str,
    inner_material: str = "Air",
):
    shell = BoxVolume(
        name=f"{name}_outer",
        mother=mother,
        size=outer_size_mm,
        translation=translation_mm,
        material=material,
    )
    sim.add_volume(shell, name=shell.name)
    opening_box_size = np.array(
        [
            inner_size_mm[0],
            inner_size_mm[1],
            outer_size_mm[2] - inner_size_mm[2] - spacing_size_mm[2],
        ]
    )
    opening_box = BoxVolume(
        name=f"{name}_opening",
        mother=shell.name,
        size=opening_box_size,
        translation=[0, 0, (inner_size_mm[2] + spacing_size_mm[2]) * 0.5],
        material=inner_material,
    )
    sim.add_volume(opening_box, name=opening_box.name)
    cavity = BoxVolume(
        name=f"{name}_cavity",
        mother=shell.name,
        size=inner_size_mm,
        translation=inner_shift_mm,
        material=inner_material,
    )
    sim.add_volume(cavity, name=cavity.name)


    return shell, cavity


def make_gate_shell_trd(
    sim: gate.Simulation,
    *,
    name: str,
    mother: str,
    top_inner_mm: float,
    top_outer_mm: float,
    bottom_inner_mm: float,
    bottom_outer_mm: float,
    length_mm: float,
    translation_mm: np.ndarray,
    material: str,
    inner_material: str = "Air",
):
    shell = TrdVolume(
        name=f"{name}_outer",
        mother=mother,
        dx1=bottom_outer_mm * 0.5,
        dy1=bottom_outer_mm * 0.5,
        dx2=top_outer_mm * 0.5,
        dy2=top_outer_mm * 0.5,
        dz=length_mm * 0.5,
        translation=translation_mm,
        material=material,
    )
    sim.add_volume(shell, name=shell.name)

    cavity = TrdVolume(
        name=f"{name}_cavity",
        mother=shell.name,
        dx1=bottom_inner_mm * 0.5,
        dy1=bottom_inner_mm * 0.5,
        dx2=top_inner_mm * 0.5,
        dy2=top_inner_mm * 0.5,
        dz=length_mm * 0.5,
        translation=[0, 0, 0],
        material=inner_material,
    )
    sim.add_volume(cavity, name=cavity.name)
    return shell, cavity


def add_pixelated_detector_to_gate_sim(
    sim: gate.Simulation,
    *,
    mother: str,
    id: int,
    translation_mm: list[float] | np.ndarray,
    pixel_size_mm: np.ndarray,
    pixel_count: np.ndarray,
):

    detector_pixel = BoxVolume(
        name=f"DetectorPixel_{id}",
        mother=mother,
        size=list(pixel_size_mm),
        translation=translation_mm,
        material="CsI",
    )
    sim.add_volume(detector_pixel, name=detector_pixel.name)

    pixel_repeater = RepeatParametrisedVolume(repeated_volume=detector_pixel)
    pixel_repeater.linear_repeat = list(pixel_count)
    pixel_repeater.translation = list(pixel_size_mm)
    sim.volume_manager.add_volume(pixel_repeater)
    pixel_repeater.material = "CsI"

def get_module_rotation_matrix(geometry_config_df: pd.DataFrame, id: int):

    # rotate around x axis by 90 degrees
    rx_0 = Rotation.from_euler("x", -90, degrees=True).as_matrix()
    # Then rotate around z axis by the azimuthal angle
    rz_1 = Rotation.from_euler(
        "z", geometry_config_df["z_axis_angle (deg)"][id] - 90, degrees=True
    ).as_matrix()
    # Then rotate around y axis by the polar angle
    rx_1 = Rotation.from_euler(
        "x", geometry_config_df["x_axis_angle (deg)"][id], degrees=True
    ).as_matrix()
    # r = rx_1 @ rx_0
    r = rz_1 @ rx_1 @ rx_0
    return r


In [124]:
sim = gate.Simulation()
sim.volume_manager.add_material_database(
    project_root / "persistent_data" / "GateMaterials.db"
)
sim.user_info.visu = True
sim.user_info.visu_type = "vrml_file_only"
sim.visu_commands_vrml = ["/vis/open VRML2FILE", "/vis/drawVolume"]
sim.visu_commands_vrml.append("/vis/geometry/set/visibility world 0 false")

crystal_size_mm = np.array([50.0, 50.0, 10.0])
pixel_count = np.array([1, 1, 1])

for module_id in range(80):
    # Get the rotation matrix for the current module
    rotation_matrix = get_module_rotation_matrix(geometry_config_df, module_id)

    # Create the collimator container volume
    collimator = TrdVolume(
        name=f"Collimator_{module_id}",
        mother="world",
        dx1=geometry_config_df["guide_outer_bottom (mm)"][module_id] * 0.5,
        dy1=geometry_config_df["guide_outer_bottom (mm)"][module_id] * 0.5,
        dx2=35,
        dy2=35,
        dz=collimator_container_l[module_id] * 0.5,
        translation=collimator_container_centers[module_id],
        rotation=rotation_matrix,
        material="Air",
    )
    sim.add_volume(collimator, name=collimator.name)
    # Add the collimator body
    make_gate_shell_trd(
        sim,
        name=f"Collimator_Body_{module_id}",
        mother=collimator.name,
        top_inner_mm=geometry_config_df["body_inner_top_corrected (mm)"][module_id],
        top_outer_mm=geometry_config_df["body_outer_top (mm)"][module_id],
        bottom_inner_mm=geometry_config_df["body_inner_bottom (mm)"][module_id],
        bottom_outer_mm=geometry_config_df["body_outer_bottom (mm)"][module_id],
        length_mm=geometry_config_df["body_l_corrected (mm)"][module_id],
        translation_mm=np.array(
            [
                0,
                0,
                collimator_body_l[module_id] * 0.5
                + collimator_guide_l[module_id]
                - collimator_container_l[module_id] * 0.5,
            ]
        ),
        material="Tungsten",
        inner_material="Air",
    )
    # Add the collimator guide
    make_gate_shell_trd(
        sim,
        name=f"Collimator_Guide_{module_id}",
        mother=collimator.name,
        top_inner_mm=geometry_config_df["guide_inner_top (mm)"][module_id],
        top_outer_mm=geometry_config_df["guide_outer_top (mm)"][module_id],
        bottom_inner_mm=geometry_config_df["guide_inner_bottom (mm)"][module_id],
        bottom_outer_mm=geometry_config_df["guide_outer_bottom (mm)"][module_id],
        length_mm=geometry_config_df["guide_l (mm)"][module_id],
        translation_mm=np.array(
            [
                0,
                0,
                -(
                    collimator_container_l[module_id]
                    - geometry_config_df["guide_l (mm)"][module_id]
                )
                * 0.5,
            ]
        ),
        material="Tungsten",
        inner_material="Air",
    )

    # Add the shielding box

    shell, cavity = make_gate_shell_box(
        sim,
        name="Shielding_Box_" + str(module_id),
        mother=collimator.name,
        outer_size_mm=geometry_config_df[
            ["box_outer_size_x (mm)", "box_outer_size_y (mm)", "box_outer_size_z (mm)"]
        ]
        .iloc[module_id]
        .to_numpy(),
        inner_size_mm=geometry_config_df[
            ["box_inner_size_x (mm)", "box_inner_size_y (mm)", "box_inner_size_z (mm)"]
        ]
        .iloc[module_id]
        .to_numpy(),
        spacing_size_mm=np.array(
            [0, 0, geometry_config_df["wall_thickness (mm)"][module_id]]
        ),
        translation_mm=np.array(
            [
                0,
                0,
                collimator_container_l[module_id] * 0.5
                - geometry_config_df["box_outer_size_z (mm)"][module_id] * 0.5,
            ]
        ),
        inner_shift_mm=np.array(
            [
                0,
                0,
                -(
                    geometry_config_df["box_outer_size_z (mm)"][module_id]
                    - geometry_config_df["box_inner_size_z (mm)"][module_id]
                )
                * 0.5
                + geometry_config_df["wall_thickness (mm)"][module_id],
            ]
        ),
        material="Tungsten",
        inner_material="Air",
    )

    frustum_cavity = TrdVolume(
        name=f"box_f_cavity_{module_id}",
        mother=shell.name,
        dx1=geometry_config_df["body_inner_top_corrected (mm)"][module_id] * 0.5,
        dy1=geometry_config_df["body_inner_top_corrected (mm)"][module_id] * 0.5,
        dx2=geometry_config_df["body_inner_top (mm)"][module_id] * 0.5,
        dy2=geometry_config_df["body_inner_top (mm)"][module_id] * 0.5,
        dz=geometry_config_df["wall_thickness (mm)"][module_id] * 0.5,
        translation=[
            0,
            0,
            (
                geometry_config_df["wall_thickness (mm)"][module_id]
                - geometry_config_df["box_outer_size_z (mm)"][module_id]
            )
            * 0.5,
        ],
        material="Air",
    )
    sim.add_volume(frustum_cavity, name=frustum_cavity.name)

    add_pixelated_detector_to_gate_sim(
        sim,
        mother=cavity.name,
        id=module_id,
        translation_mm=np.array(
            [
                0,
                0,
                (
                    crystal_size_mm[2]
                    - geometry_config_df["box_inner_size_z (mm)"][module_id]
                ),
            ]
        ),
        pixel_count=pixel_count,
        pixel_size_mm=crystal_size_mm / pixel_count,
    )


visu_filename = "test_cardiac_spect_geometry.wrl"
sim.visu_commands_vrml.append("/vis/viewer/flush")
sim.user_info.visu_filename = str(Path(visu_filename).resolve())
sim.run(start_new_process=True)

Dispatching simulation to subprocess ...
⚠️ No configured source, no particle will be generated.
Simulation: create RunManager (single thread)
Simulation: initialize Geometry
Simulation: initialize Physics
Simulation: initialize Sources
Simulation: initialize Auxiliary attributes
Simulation: initialize Visualization
Simulation: initialize Actors
Simulation: initialize G4RunManager
⚠️ G4Exception origin: G4PVPlacement::CheckOverlaps()
G4Exception code: GeomVol1002
G4Exception severity: G4ExceptionSeverity.JustWarning
G4Exception: Overlap with volume already placed !
          Overlap is detected for volume Collimator_1:0 (G4Trd) with Collimator_0:0 (G4Trd)
          overlap at local point (-24.769,19.1001,27.5731) by 2.6848 mm  (max of 18 cases)
NOTE: Reached maximum fixed number -1- of overlaps reports for this volume !
⚠️ G4Exception origin: G4PVPlacement::CheckOverlaps()
G4Exception code: GeomVol1002
G4Exception severity: G4ExceptionSeverity.JustWarning
G4Exception: Overlap with volu

In [125]:
from qmirt.plot.wrl import plot_wrl_file

fig = plot_wrl_file("test_cardiac_spect_geometry.wrl")
fig.add_traces(
    [
        go.Scatter3d(
            x=hole_centers[:1, 0],
            y=hole_centers[:1, 1],
            z=hole_centers[:1, 2],
            mode="markers",
            marker=dict(size=1, color="blue", opacity=0.8),
            name="Hole Centers",
        ),
    ]
)

fig.show()